In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, accuracy_score, f1_score, confusion_matrix, precision_score, recall_score, classification_report,roc_auc_score,roc_curve,RocCurveDisplay
import os
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
os.chdir('/home/pgcp-ai/MachineLearning/Datasets/')

In [2]:
sonar = pd.read_csv("Sonar.csv")
sonar

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V52,V53,V54,V55,V56,V57,V58,V59,V60,Class
0,0.0200,0.0371,0.0428,0.0207,0.0954,0.0986,0.1539,0.1601,0.3109,0.2111,...,0.0027,0.0065,0.0159,0.0072,0.0167,0.0180,0.0084,0.0090,0.0032,R
1,0.0453,0.0523,0.0843,0.0689,0.1183,0.2583,0.2156,0.3481,0.3337,0.2872,...,0.0084,0.0089,0.0048,0.0094,0.0191,0.0140,0.0049,0.0052,0.0044,R
2,0.0262,0.0582,0.1099,0.1083,0.0974,0.2280,0.2431,0.3771,0.5598,0.6194,...,0.0232,0.0166,0.0095,0.0180,0.0244,0.0316,0.0164,0.0095,0.0078,R
3,0.0100,0.0171,0.0623,0.0205,0.0205,0.0368,0.1098,0.1276,0.0598,0.1264,...,0.0121,0.0036,0.0150,0.0085,0.0073,0.0050,0.0044,0.0040,0.0117,R
4,0.0762,0.0666,0.0481,0.0394,0.0590,0.0649,0.1209,0.2467,0.3564,0.4459,...,0.0031,0.0054,0.0105,0.0110,0.0015,0.0072,0.0048,0.0107,0.0094,R
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
203,0.0187,0.0346,0.0168,0.0177,0.0393,0.1630,0.2028,0.1694,0.2328,0.2684,...,0.0116,0.0098,0.0199,0.0033,0.0101,0.0065,0.0115,0.0193,0.0157,M
204,0.0323,0.0101,0.0298,0.0564,0.0760,0.0958,0.0990,0.1018,0.1030,0.2154,...,0.0061,0.0093,0.0135,0.0063,0.0063,0.0034,0.0032,0.0062,0.0067,M
205,0.0522,0.0437,0.0180,0.0292,0.0351,0.1171,0.1257,0.1178,0.1258,0.2529,...,0.0160,0.0029,0.0051,0.0062,0.0089,0.0140,0.0138,0.0077,0.0031,M
206,0.0303,0.0353,0.0490,0.0608,0.0167,0.1354,0.1465,0.1123,0.1945,0.2354,...,0.0086,0.0046,0.0126,0.0036,0.0035,0.0034,0.0079,0.0036,0.0048,M


In [3]:
le = LabelEncoder()
sonar["Class"] = le.fit_transform(sonar["Class"])


In [4]:
X, y = sonar.drop("Class", axis = 1), sonar["Class"]

In [5]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,stratify=y,random_state=26)

In [6]:
lda = LinearDiscriminantAnalysis()
lda.fit(X_train,y_train) # Calculates all the apriori probabilities and 
y_pred_prob = lda.predict_proba(X_test) # Calculates the posterior probabilites

In [7]:
y_pred = lda.predict(X_test)

In [8]:
print(f"ROC-AUC Score is: {roc_auc_score(y_test,y_pred_prob[:,1])}")

ROC-AUC Score is: 0.7941176470588235


In [9]:
print(f"Accuracy Score: {accuracy_score(y_test, y_pred)}")

Accuracy Score: 0.7301587301587301


### Quadratic Discriminant Analysis

In [10]:
qda = QuadraticDiscriminantAnalysis(reg_param=0.8)
qda.fit(X_train,y_train) # Calculates all the apriori probabilities and 
y_pred_prob = qda.predict_proba(X_test) # Calculates the posterior probabilites

In [11]:
y_pred = qda.predict(X_test)
y_pred

array([1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1,
       1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0])

In [12]:
print(f"ROC-AUC Score is: {roc_auc_score(y_test,y_pred_prob[:,1])}")

ROC-AUC Score is: 0.8255578093306288


In [13]:
print(f"Accuracy Score: {accuracy_score(y_test, y_pred)}")

Accuracy Score: 0.7142857142857143


In [18]:
scores = []
for i in np.linspace(0.001,1):
    qda = QuadraticDiscriminantAnalysis(reg_param=i)
    qda.fit(X_train,y_train) # Calculates all the apriori probabilities and 
    y_pred_prob = qda.predict_proba(X_test) # Calculates the posterior probabilites
    y_pred = qda.predict(X_test)
    scores.append([i,log_loss(y_test,y_pred_prob),accuracy_score(y_test,y_pred)])
df_scores = pd.DataFrame(scores,columns=['Reg_Param','Log Loss Score','Accuracy Score'])
df_scores.sort_values(['Log Loss Score','Accuracy Score'],ascending=[True,False])

,Reg_Param,Log Loss Score,Accuracy Score
2,0.041776,0.369842,0.857143
3,0.062163,0.390233,0.841270
1,0.021388,0.392692,0.873016
4,0.082551,0.415371,0.857143
5,0.102939,0.439099,0.857143
6,0.123327,0.460239,0.857143
7,0.143714,0.478767,0.857143
8,0.164102,0.494954,0.841270
9,0.184490,0.509123,0.841270
10,0.204878,0.521573,0.825397
